In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl

sys.path.append('../src')

from data_pipeline import load_all_raw_data

from data_analysis import (
    filter_data_until_date, temporal_split_data, 
)

from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)

from preprocess import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)

from weather_data import WeatherDataCollector


In [ ]:
df_feat = pl.read_parquet(r"C:\Users\xxx\Documents\GitHub\EcoBici-AI\notebooks\feature_checkpoints\df_feat_final_result.parquet")

In [ ]:
# Check if GPU is available
import xgboost as xgb
print("\nChecking GPU availability for XGBoost...")
try:
    # Build a small test DMatrix
    import numpy as np
    test_data = np.random.rand(10, 10)
    test_label = np.random.rand(10)
    test_dmatrix = xgb.DMatrix(test_data, label=test_label)
    
    # Try to train a small model with GPU
    test_params = {'tree_method': 'gpu_hist'}
    test_bst = xgb.train(test_params, test_dmatrix, num_boost_round=1)
    print("✓ GPU is available and working")
    use_gpu = True
except Exception as e:
    print("✗ GPU not available:", str(e))
    print("Falling back to CPU training")
    use_gpu = False

feature_cols = ['station_id', 'dep_last_DT', 'trip_dur_mean_last_DT',
       'model_FIT_cnt', 'model_ICONIC_cnt', 'share_male', 'share_female',
       'share_other', 'dep_lag_1', 'dep_lag_2', 'dep_lag_3', 'dep_lag_4',
       'dep_lag_5', 'dep_lag_6', 'arr_last_DT', 'arr_lag_1', 'arr_lag_2',
       'arr_lag_3', 'arr_lag_4', 'arr_lag_5', 'arr_lag_6', 'weather_temperature_2m',
       'weather_relative_humidity_2m', 'weather_dew_point_2m',
       'weather_apparent_temperature', 'weather_precipitation', 'weather_rain',
       'weather_pressure_msl', 'weather_surface_pressure', 'weather_cloud_cover',
       'weather_cloud_cover_low', 'weather_cloud_cover_mid',
       'weather_cloud_cover_high', 'weather_et0_fao_evapotranspiration',
       'weather_vapour_pressure_deficit',
       'weather_soil_temperature_0_to_7cm',
       'weather_soil_moisture_0_to_7cm', 'weather_sunshine_duration',
       'weather_is_day', 'weather_direct_radiation', 'precip_flag',
       'wind_dir_sin', 'wind_dir_cos', 'weather_code_cat_Clear',
       'weather_code_cat_Clouds', 'weather_code_cat_Drizzle',
       'weather_code_cat_Rain', 'weather_code_cat_null',
       'precipitation_lag_1h', 'rain_lag_1h', 'is_holiday_ar', 'sin_hour',
       'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month',
       'is_weekend', 'payday_flag', 'vacation_season', 'peak_commute',
       'dep_ma_24h', 'dep_std_24h', 'dep_ratio_DT_24h', 'near_dep_sum_DT',
       'near_dep_lag_1']

# Split data temporally for training and validation
print("\nSplitting data temporally...")
split_date = pl.datetime(2023, 12, 1)  # Fixed datetime constructor call
train_data = df_feat.filter(pl.col("ts_start") < split_date)
val_data = df_feat.filter(pl.col("ts_start") >= split_date)

# Define features and target
target_col = 'y_arrivals_next_DT'
# Train XGBoost model with chunking
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set XGBoost parameters
params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}

# Add GPU parameters if available
if use_gpu:
    params.update({
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'gpu_id': 0
    })
else:
    params.update({
        'tree_method': 'hist'
    })

# Initialize model
model = xgb.XGBRegressor(**params)

# Process data in chunks
chunk_size = 1_000_000  # Adjust based on available memory

print(f"\nTraining XGBoost model in chunks using {'GPU' if use_gpu else 'CPU'}...")
for i in range(0, len(train_data), chunk_size):
    # Get chunk of data
    chunk_end = min(i + chunk_size, len(train_data))
    print(f"Processing chunk {i//chunk_size + 1}, rows {i} to {chunk_end}")
    
    # Convert chunk to pandas
    train_chunk = train_data[i:chunk_end].to_pandas()
    
    # Update model with this chunk
    model.fit(
        train_chunk[feature_cols],
        train_chunk[target_col],
        xgb_model=model.get_booster() if i > 0 else None
    )

print("\nMaking predictions...")
# Process validation data in chunks too
val_pred = []
val_true = []
for i in range(0, len(val_data), chunk_size):
    chunk_end = min(i + chunk_size, len(val_data))
    val_chunk = val_data[i:chunk_end].to_pandas()
    
    chunk_pred = model.predict(val_chunk[feature_cols])
    val_pred.extend(chunk_pred)
    val_true.extend(val_chunk[target_col].values)

# Evaluate model
print("\nValidation metrics:")
print(f"MSE: {mean_squared_error(val_true, val_pred):.4f}")
print(f"MAE: {mean_absolute_error(val_true, val_pred):.4f}")
print(f"R2: {r2_score(val_true, val_pred):.4f}")

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features:")
print(importance.head(10))

# Save model
model.save_model('xgb_model.json')
print("\nModel saved as 'xgb_model.json'")
